# 03. TF-IDF and Resume-JD Similarity

This notebook uses the actual processed synthetic dataset to demonstrate numerical text representation and ranking.

## 1. Introduction

TF-IDF represents words numerically while reducing the influence of terms that appear in every document.

## 2. Why numerical text representation is needed

Machine-learning algorithms cannot compare raw paragraphs directly. Vectors allow mathematical comparison while preserving important vocabulary.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

resumes = pd.read_csv(ROOT / "data" / "processed" / "resumes_processed.csv")
jds = pd.read_csv(ROOT / "data" / "processed" / "job_descriptions_processed.csv")
print("Resumes:", resumes.shape, "JDs:", jds.shape)

Resumes: (50, 10) JDs: (20, 9)


## 3. Simple document collection

In [2]:
documents = [
    "python sql data analysis",
    "python machine learning model",
    "java web development"
]
print(documents)

['python sql data analysis', 'python machine learning model', 'java web development']


## 4. TF-IDF vectorization

In [3]:
small_vectorizer = TfidfVectorizer()
small_matrix = small_vectorizer.fit_transform(documents)
print("Matrix shape:", small_matrix.shape)

Matrix shape: (3, 10)


## 5. Display feature/vocabulary names

In [4]:
print(small_vectorizer.get_feature_names_out().tolist())
print(small_matrix.toarray().round(3))

['analysis', 'data', 'development', 'java', 'learning', 'machine', 'model', 'python', 'sql', 'web']
[[0.529 0.529 0.    0.    0.    0.    0.    0.402 0.529 0.   ]
 [0.    0.    0.    0.    0.529 0.529 0.529 0.402 0.    0.   ]
 [0.    0.    0.577 0.577 0.    0.    0.    0.    0.    0.577]]


## 6. Display or explain the TF-IDF matrix

Rows represent documents and columns represent vocabulary terms. A larger value means that a term is more informative for that document relative to this collection.

In [5]:
small_matrix_frame = pd.DataFrame(small_matrix.toarray().round(3), columns=small_vectorizer.get_feature_names_out())
small_matrix_frame

,analysis,data,development,java,learning,machine,model,python,sql,web
0,0.529,0.529,0.000,0.000,0.000,0.000,0.000,0.402,0.529,0.000
1,0.000,0.000,0.000,0.000,0.529,0.529,0.529,0.402,0.000,0.000
2,0.000,0.000,0.577,0.577,0.000,0.000,0.000,0.000,0.000,0.577


## 7. Resume representation

In [6]:
resume_texts = resumes["cleaned_text"].fillna("").tolist()
print("First resume vector input:", resume_texts[0])

First resume vector input: synthetic data analyst candidate with skills python sql and 2 years of experience.


## 8. Job-description representation

In [7]:
job_text = jds.loc[0, "cleaned_text"]
print("Selected JD:", job_text)

Selected JD: synthetic data analyst position requiring python sql.


## 9. Cosine similarity explanation

Cosine similarity compares the angle between vectors. Here it provides a simple lexical overlap signal; it is not a validated hiring score.

In [8]:
texts = [job_text] + resume_texts
vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(texts)
similarity_scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()
print("Score range:", similarity_scores.min(), "to", similarity_scores.max())

Score range: 0.022238319302091233 to 0.511555269077924


## 10. Calculate resume-JD similarity

In [9]:
score_frame = pd.DataFrame({
    "candidate_label": resumes["candidate_label"],
    "similarity": similarity_scores,
})
score_frame.head()

,candidate_label,similarity
0,Candidate_001,0.511555
1,Candidate_002,0.122776
2,Candidate_003,0.122776
3,Candidate_004,0.022238
4,Candidate_005,0.024583


## 11. Compare multiple candidates

In [10]:
print(score_frame["similarity"].describe())
print("Candidates with score above the median:", (score_frame["similarity"] > score_frame["similarity"].median()).sum())

count    50.000000
mean      0.160786
std       0.182765
min       0.022238
25%       0.024583
50%       0.122776
75%       0.122776
max       0.511555
Name: similarity, dtype: float64
Candidates with score above the median: 10


## 12. Rank candidates

In [11]:
from src.similarity_demo import rank
ranked = rank(resumes, job_text, top_k=10)
ranked

,candidate_label,similarity
0,Candidate_001,0.511555
1,Candidate_006,0.511555
2,Candidate_016,0.511555
3,Candidate_011,0.511555
4,Candidate_046,0.511555
5,Candidate_041,0.511555
6,Candidate_036,0.511555
7,Candidate_031,0.511555
8,Candidate_026,0.511555
9,Candidate_021,0.511555


## 13. Display top candidates in a table

In [12]:
top_candidates = ranked.merge(resumes[["candidate_label", "job_category", "skills"]], on="candidate_label")
top_candidates

,candidate_label,similarity,job_category,skills
0,Candidate_001,0.511555,Data Analyst,Python;SQL
1,Candidate_006,0.511555,Data Analyst,Python;SQL
2,Candidate_016,0.511555,Data Analyst,Python;SQL
3,Candidate_011,0.511555,Data Analyst,Python;SQL
4,Candidate_046,0.511555,Data Analyst,Python;SQL
5,Candidate_041,0.511555,Data Analyst,Python;SQL
6,Candidate_036,0.511555,Data Analyst,Python;SQL
7,Candidate_031,0.511555,Data Analyst,Python;SQL
8,Candidate_026,0.511555,Data Analyst,Python;SQL
9,Candidate_021,0.511555,Data Analyst,Python;SQL


## 14. Meaning of the similarity score

A higher value means greater TF-IDF vocabulary alignment with this selected JD. It does not prove skills, experience, fairness, suitability, or hiring potential.

## 15. Educational baseline warning

This is an educational Month-1 baseline and **not a hiring decision**, recruiter recommendation, or validated production model.

## 16. Limitations of TF-IDF

- It does not understand synonyms or context.
- It can favor repeated or shared vocabulary.
- It does not verify claims in a resume.
- Results depend on the document collection and preprocessing choices.
- Evaluation, fairness review, and human oversight would be required for later work.

## 17. Key Learning Outcomes

- Convert text into TF-IDF vectors.
- Inspect vocabulary and matrix values.
- Calculate cosine similarity.
- Compare and rank synthetic candidates while interpreting limitations responsibly.